In [1]:
from functools import partial

import jax
import jax.random as jr
import jax_healpy as jhp
from jax.sharding import AxisType, NamedSharding
from jax.sharding import PartitionSpec as P

nside = 32
key = jr.key(0)

mesh = jax.make_mesh((8, 1), ("x", "y"), axis_types=(AxisType.Auto, AxisType.Auto))
sharding = NamedSharding(mesh, P("x", "y"))
print(f"Mesh shape     : {mesh.shape}")
print(f"Partition spec : {sharding.spec}")

/lustre/fswork/projects/rech/tkc/commun/venv/pm/lib/python3.12/site-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.10.1 is installed, but it is not compatible with the installed jaxlib version 0.10.0, so it will not be used.
  warnings.warn(


JAX is not using 64-bit precision. This will dramatically affect numerical precision at even moderate L.


JAX is not using 64-bit precision. Results will diverge from healpy even at moderate nside and can overflow for nside > 8192. See the README on numerical precision.


Mesh shape     : OrderedDict({'x': 8, 'y': 1})
Partition spec : P('x', 'y')


In [2]:
npix = jhp.nside2npix(nside)
hp_map = jax.random.normal(
    key,
    (
        8,
        npix,
    ),
)
hp_map = jax.lax.with_sharding_constraint(hp_map, sharding)


def sharded_map2alm(hp_map, method):
    partialled = partial(jhp.map2alm, method=method)
    return jax.shard_map(partialled, mesh=mesh, in_specs=P("x", None), out_specs=P("x", None))(hp_map)


jax.debug.visualize_array_sharding(hp_map)

                                     GPU 0                                      
                                                                                
                                     GPU 1                                      
                                                                                
                                     GPU 2                                      
                                                                                
                                     GPU 3                                      
                                                                                
                                     GPU 4                                      
                                                                                
                                     GPU 5                                      
                                                                                
                                     GPU 6                                      
                                                                                
                                     GPU 7                                      
                                                                                

In [3]:
jax.clear_caches()

In [4]:
%time cuda_alms = sharded_map2alm(hp_map , method="jax_cuda").block_until_ready()

In [ ]:
%time jax_alms = sharded_map2alm(hp_map , method="jax").block_until_ready()

In [ ]:
%timeit cuda_alms = sharded_map2alm(hp_map , method="jax_cuda").block_until_ready()

In [ ]:
%timeit jax_alms = sharded_map2alm(hp_map , method="jax").block_until_ready()

In [ ]:
print("Finished all good")